# Transformer + Linear Atomic-Sum M3GNet 冒烟测试

本分支只做一个受控改动：`TransformerEncoder -> Linear(64, 1) -> 按图求和`。M3GNet blocks、Transformer 配置、Potential 和损失逻辑保持不变。

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matgl
import torch

print('Repository:', repo_root)
print('MatGL source:', matgl.__file__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
from torch import nn
from matgl.models._m3gnet import M3GNet

model = M3GNet(
    element_types=('Si',),
    is_intensive=False,
    readout_type='transformer',
    transformer_nhead=4,
    transformer_num_layers=1,
    transformer_dim_ff=128,
    transformer_dropout=0.0,
)

assert isinstance(model.final_layer.atomic_head, nn.Linear)
print('Final layer:', type(model.final_layer).__name__)
print('Atomic head:', type(model.final_layer.atomic_head).__name__)
print('Head weight shape:', tuple(model.final_layer.atomic_head.weight.shape))
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 势能前向与反向传播

下面同时检查能量、力、应力以及二阶自动微分。`SDPBackend.MATH` 用于支持力损失所需的高阶梯度。

In [ ]:
from pymatgen.core import Lattice, Structure
from torch.nn.attention import SDPBackend, sdpa_kernel
from matgl.apps.pes import Potential
from matgl.ext.pymatgen import Structure2Graph

structure = Structure(
    Lattice.cubic(5.43),
    ['Si', 'Si'],
    [[0, 0, 0], [0.25, 0.25, 0.25]],
)
potential = Potential(model=model, calc_forces=True, calc_stresses=True)
converter = Structure2Graph(element_types=('Si',), cutoff=model.cutoff)
graph, lattice, state_attr = converter.get_graph(structure)
state_attr = torch.tensor(state_attr, dtype=torch.float32)

model.train()
potential.train()
with sdpa_kernel(SDPBackend.MATH):
    energy, forces, stresses, _ = potential(
        g=graph, lat=lattice, state_attr=state_attr
    )
    smoke_loss = (
        energy.square().mean()
        + forces.square().mean()
        + stresses.square().mean()
    )
    smoke_loss.backward()

head_grad = model.final_layer.atomic_head.weight.grad
attention_grad = model.final_layer.transformer.layers[0].self_attn.in_proj_weight.grad
assert torch.isfinite(energy).all()
assert torch.isfinite(forces).all()
assert torch.isfinite(stresses).all()
assert torch.isfinite(head_grad).all()
assert torch.isfinite(attention_grad).all()
print('Energy:', float(energy.detach()))
print('Forces shape:', tuple(forces.shape))
print('Stress shape:', tuple(stresses.shape))
print('Smoke loss:', float(smoke_loss.detach()))
print('Linear-head gradient norm:', float(head_grad.norm()))
print('Transformer gradient norm:', float(attention_grad.norm()))
print('PASS: forward, forces/stresses, and backward are finite.')